In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Crypto_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/bitcoin_1m.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20)

df.printSchema()

Raw Dataset Loaded
Total Rows: 1000
+-------------------+--------+--------+--------+--------+----------+
|          timestamp|    open|    high|     low|   close|    volume|
+-------------------+--------+--------+--------+--------+----------+
|2026-06-10 17:17:00|61154.98|61154.98| 61106.8| 61106.8|0.06467535|
|2026-06-10 17:18:00| 61102.0|61138.77| 61102.0|61121.64|0.07313276|
|2026-06-10 17:19:00|61109.01|61160.77|61109.01| 61147.9|1.83580275|
|2026-06-10 17:20:00|61138.43|61138.43|61113.63|61113.63|0.42116973|
|2026-06-10 17:21:00|61109.01| 61182.0| 61107.4|61176.53|0.51739396|
|2026-06-10 17:22:00| 61178.1|61181.76|61166.31|61181.76|0.09351303|
|2026-06-10 17:23:00|61183.21|61210.75|61171.25|61210.75|0.13678772|
|2026-06-10 17:24:00|61211.15|61230.03|61200.86|61210.01|0.10933518|
|2026-06-10 17:25:00|61220.98|61226.65|61211.01| 61213.2|0.17481438|
|2026-06-10 17:26:00|61209.26|61209.26|61179.74|61179.74|0.72710502|
|2026-06-10 17:27:00|61197.01|61257.06|61197.01| 61256.0|0.22554629

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+---------+----+----+---+-----+------+
|timestamp|open|high|low|close|volume|
+---------+----+----+---+-----+------+
|        0|   0|   0|  0|    0|     0|
+---------+----+----+---+-----+------+

Total Rows: 1000
Unique Rows: 1000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df = df.orderBy("timestamp")

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2026-06-10 17:17:00|2026-06-11 09:56:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("prev_time", F.lag("timestamp").over(w))

df = df.withColumn(
    "diff_min",
    (F.unix_timestamp("timestamp")
     - F.unix_timestamp("prev_time")) / 60
)

df.select(
    "timestamp",
    "prev_time",
    "diff_min"
).show(20, False)

gap_count = df.filter(F.col("diff_min") > 1).count()

print("Gap Count:", gap_count)

+-------------------+-------------------+--------+
|timestamp          |prev_time          |diff_min|
+-------------------+-------------------+--------+
|2026-06-10 17:17:00|NULL               |NULL    |
|2026-06-10 17:18:00|2026-06-10 17:17:00|1.0     |
|2026-06-10 17:19:00|2026-06-10 17:18:00|1.0     |
|2026-06-10 17:20:00|2026-06-10 17:19:00|1.0     |
|2026-06-10 17:21:00|2026-06-10 17:20:00|1.0     |
|2026-06-10 17:22:00|2026-06-10 17:21:00|1.0     |
|2026-06-10 17:23:00|2026-06-10 17:22:00|1.0     |
|2026-06-10 17:24:00|2026-06-10 17:23:00|1.0     |
|2026-06-10 17:25:00|2026-06-10 17:24:00|1.0     |
|2026-06-10 17:26:00|2026-06-10 17:25:00|1.0     |
|2026-06-10 17:27:00|2026-06-10 17:26:00|1.0     |
|2026-06-10 17:28:00|2026-06-10 17:27:00|1.0     |
|2026-06-10 17:29:00|2026-06-10 17:28:00|1.0     |
|2026-06-10 17:30:00|2026-06-10 17:29:00|1.0     |
|2026-06-10 17:31:00|2026-06-10 17:30:00|1.0     |
|2026-06-10 17:32:00|2026-06-10 17:31:00|1.0     |
|2026-06-10 17:33:00|2026-06-10

In [7]:
# =========================================
# MA10 & MA60
# =========================================

w10 = Window.orderBy("timestamp").rowsBetween(-9, 0)
w60 = Window.orderBy("timestamp").rowsBetween(-59, 0)

df = df.withColumn("MA10", F.avg("close").over(w10))
df = df.withColumn("MA60", F.avg("close").over(w60))

df.select(
    "timestamp",
    "close",
    "MA10",
    "MA60"
).show(20, False)

+-------------------+--------+------------------+------------------+
|timestamp          |close   |MA10              |MA60              |
+-------------------+--------+------------------+------------------+
|2026-06-10 17:17:00|61106.8 |61106.8           |61106.8           |
|2026-06-10 17:18:00|61121.64|61114.22          |61114.22          |
|2026-06-10 17:19:00|61147.9 |61125.44666666666 |61125.44666666666 |
|2026-06-10 17:20:00|61113.63|61122.4925        |61122.4925        |
|2026-06-10 17:21:00|61176.53|61133.3           |61133.3           |
|2026-06-10 17:22:00|61181.76|61141.37666666667 |61141.37666666667 |
|2026-06-10 17:23:00|61210.75|61151.287142857145|61151.287142857145|
|2026-06-10 17:24:00|61210.01|61158.6275        |61158.6275        |
|2026-06-10 17:25:00|61213.2 |61164.69111111111 |61164.69111111111 |
|2026-06-10 17:26:00|61179.74|61166.195999999996|61166.195999999996|
|2026-06-10 17:27:00|61256.0 |61181.116         |61174.35999999999 |
|2026-06-10 17:28:00|61290.02|6119

In [8]:
# =========================================
# ROC + MOMENTUM
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("close_lag10", F.lag("close", 10).over(w))

df = df.withColumn(
    "ROC",
    (F.col("close") - F.col("close_lag10"))
    / F.col("close_lag10") * 100
)

df = df.withColumn(
    "MOM",
    F.col("close") - F.col("close_lag10")
)

df.select(
    "close",
    "close_lag10",
    "ROC",
    "MOM"
).show(20, False)

+--------+-----------+-------------------+------------------+
|close   |close_lag10|ROC                |MOM               |
+--------+-----------+-------------------+------------------+
|61106.8 |NULL       |NULL               |NULL              |
|61121.64|NULL       |NULL               |NULL              |
|61147.9 |NULL       |NULL               |NULL              |
|61113.63|NULL       |NULL               |NULL              |
|61176.53|NULL       |NULL               |NULL              |
|61181.76|NULL       |NULL               |NULL              |
|61210.75|NULL       |NULL               |NULL              |
|61210.01|NULL       |NULL               |NULL              |
|61213.2 |NULL       |NULL               |NULL              |
|61179.74|NULL       |NULL               |NULL              |
|61256.0 |61106.8    |0.24416267911263081|149.1999999999971 |
|61290.02|61121.64   |0.2754834457975888 |168.37999999999738|
|61274.1 |61147.9    |0.20638484723105305|126.19999999999709|
|61340.4

In [9]:
# =========================================
# RSI 14
# =========================================

w1 = Window.orderBy("timestamp")
w14 = Window.orderBy("timestamp").rowsBetween(-13, 0)

df = df.withColumn("change", F.col("close") - F.lag("close").over(w1))

df = df.withColumn(
    "gain",
    F.when(F.col("change") > 0, F.col("change")).otherwise(0)
)

df = df.withColumn(
    "loss",
    F.when(F.col("change") < 0, -F.col("change")).otherwise(0)
)

df = df.withColumn("avg_gain", F.avg("gain").over(w14))
df = df.withColumn("avg_loss", F.avg("loss").over(w14))

df = df.withColumn(
    "RS",
    F.when(F.col("avg_loss") == 0, None)
     .otherwise(F.col("avg_gain") / F.col("avg_loss"))
)

df = df.withColumn(
    "RSI",
    F.when(F.col("avg_loss") == 0, 100)
     .when(F.col("avg_gain") == 0, 0)
     .otherwise(
         100 - (100 / (1 + F.col("RS")))
     )
)

print("RSI Created")

df.select(
    "timestamp",
    "close",
    "change",
    "gain",
    "loss",
    "avg_gain",
    "avg_loss",
    "RS",
    "RSI"
).show(20, False)

RSI Created
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+
|timestamp          |close   |change             |gain              |loss              |avg_gain          |avg_loss          |RS                |RSI              |
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+
|2026-06-10 17:17:00|61106.8 |NULL               |0.0               |0.0               |0.0               |0.0               |NULL              |100.0            |
|2026-06-10 17:18:00|61121.64|14.839999999996508 |14.839999999996508|0.0               |7.419999999998254 |0.0               |NULL              |100.0            |
|2026-06-10 17:19:00|61147.9 |26.260000000002037 |26.260000000002037|0.0               |13.699999999999514|0.0               |NULL              |100.0            |
|202

In [10]:
# =========================================
# STOCHASTIC OSCILLATOR
# =========================================

df = df.withColumn(
    "highest_high",
    F.max("high").over(w14)
)

df = df.withColumn(
    "lowest_low",
    F.min("low").over(w14)
)

df = df.withColumn(
    "stoch_k",
    F.when(
        (F.col("highest_high") - F.col("lowest_low")) == 0,
        None
    ).otherwise(
        (F.col("close") - F.col("lowest_low"))
        /
        (F.col("highest_high") - F.col("lowest_low"))
        * 100
    )
)

w3 = Window.orderBy("timestamp").rowsBetween(-2, 0)

df = df.withColumn(
    "stoch_d",
    F.avg("stoch_k").over(w3)
)

print("Stochastic Created")

df.select(
    "timestamp",
    "close",
    "highest_high",
    "lowest_low",
    "stoch_k",
    "stoch_d"
).show(20, False)

Stochastic Created
+-------------------+--------+------------+----------+------------------+------------------+
|timestamp          |close   |highest_high|lowest_low|stoch_k           |stoch_d           |
+-------------------+--------+------------+----------+------------------+------------------+
|2026-06-10 17:17:00|61106.8 |61154.98    |61106.8   |0.0               |0.0               |
|2026-06-10 17:18:00|61121.64|61154.98    |61102.0   |37.070592676478356|18.535296338239178|
|2026-06-10 17:19:00|61147.9 |61160.77    |61102.0   |78.10107197550444 |38.39055488399426 |
|2026-06-10 17:20:00|61113.63|61160.77    |61102.0   |19.789007997274144|44.98689088308564 |
|2026-06-10 17:21:00|61176.53|61182.0     |61102.0   |93.16249999999854 |63.68419332425904 |
|2026-06-10 17:22:00|61181.76|61182.0     |61102.0   |99.70000000000255 |70.88383599909174 |
|2026-06-10 17:23:00|61210.75|61210.75    |61102.0   |100.0             |97.62083333333369 |
|2026-06-10 17:24:00|61210.01|61230.03    |61102.0 